In [0]:
#creating some sample data

data = [ ("Alice", "UAE", 5000),
        ("Bob", "KSA", 7000),
        ("Carol", "UAE", 4500)]

In [0]:
#define column names

columns = ["name", "country", "salary"]

#create a dataframe like table in memory

df = spark.createDataFrame(data, columns)

#step 4 - show it like select * from table

df.show()

+-----+-------+------+
| name|country|salary|
+-----+-------+------+
|Alice|    UAE|  5000|
|  Bob|    KSA|  7000|
|Carol|    UAE|  4500|
+-----+-------+------+



In [0]:
#filter only UAE

uae_df = df.filter(df.country == "UAE")

uae_df.show()

+-----+-------+------+
| name|country|salary|
+-----+-------+------+
|Alice|    UAE|  5000|
|Carol|    UAE|  4500|
+-----+-------+------+



In [0]:
df.groupBy("country").count().show()

+-------+-----+
|country|count|
+-------+-----+
|    UAE|    2|
|    KSA|    1|
+-------+-----+



In [0]:
#add a new column - like CASE WHEN in teradata

from pyspark.sql.functions import when, col 

df_with_grade = df.withColumn("grade", when(col("salary")>=6000, "Senior")
                              .otherwise("Junior")
                                            )
df_with_grade.show()                                            

+-----+-------+------+------+
| name|country|salary| grade|
+-----+-------+------+------+
|Alice|    UAE|  5000|Junior|
|  Bob|    KSA|  7000|Senior|
|Carol|    UAE|  4500|Junior|
+-----+-------+------+------+



In [0]:
df_with_grade.orderBy(col("salary").desc()).show()

+-----+-------+------+------+
| name|country|salary| grade|
+-----+-------+------+------+
|  Bob|    KSA|  7000|Senior|
|Alice|    UAE|  5000|Junior|
|Carol|    UAE|  4500|Junior|
+-----+-------+------+------+



In [0]:
df_with_grade.write.format("delta").saveAsTable("employee_grades")
print("Delta table created successfully")

Delta table created successfully


In [0]:
df_loaded = spark.read.format("delta").table("employee_grades")
df_loaded.show()

+-----+-------+------+------+
| name|country|salary| grade|
+-----+-------+------+------+
|Alice|    UAE|  5000|Junior|
|  Bob|    KSA|  7000|Senior|
|Carol|    UAE|  4500|Junior|
+-----+-------+------+------+



In [0]:
#we will simulate Time Travel here

from pyspark.sql import Row

new_employee = spark.createDataFrame([Row(name="David", country="UAE", salary = 9000,
                                      grade = "Senior")])
new_employee.write.format("delta").mode("append").saveAsTable("employee_grades")    

spark.read.format("delta").table("employee_grades").show()

+-----+-------+------+------+
| name|country|salary| grade|
+-----+-------+------+------+
|Alice|    UAE|  5000|Junior|
|  Bob|    KSA|  7000|Senior|
|Carol|    UAE|  4500|Junior|
|David|    UAE|  9000|Senior|
|David|    UAE|  9000|Senior|
+-----+-------+------+------+



In [0]:
df_clean = spark.read.format("delta").table("employee_grades").dropDuplicates()
df_clean.show()

+-----+-------+------+------+
| name|country|salary| grade|
+-----+-------+------+------+
|  Bob|    KSA|  7000|Senior|
|Carol|    UAE|  4500|Junior|
|Alice|    UAE|  5000|Junior|
|David|    UAE|  9000|Senior|
+-----+-------+------+------+



In [0]:
df_clean.write.format("delta").mode("overwrite").saveAsTable("employee_grades")
spark.read.format("delta").table("employee_grades").show()

+-----+-------+------+------+
| name|country|salary| grade|
+-----+-------+------+------+
|  Bob|    KSA|  7000|Senior|
|Carol|    UAE|  4500|Junior|
|Alice|    UAE|  5000|Junior|
|David|    UAE|  9000|Senior|
+-----+-------+------+------+



In [0]:
#check what version of delta table we have 

from delta.tables import DeltaTable
delta_table = DeltaTable.forName(spark, "employee_grades")
delta_table.history().show(truncate= False)

+-------+-------------------+---------------+--------------------------+---------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+------------------------------------------------------------------------------------------------------------------------------------------+------------+------------------------------------------+
|version|timestamp          |userId         |userName                  |operation                        |operationParameters                                                                                                                                                                             |job |notebook          |queryHistoryStatementId             |clusterId      

In [0]:
delta_table.history().select( 
                             "version",
                             "timestamp",
                             "operation",
                             "operationParameters"
).show(truncate = False)

+-------+-------------------+---------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|version|timestamp          |operation                        |operationParameters                                                                                                                                                                             |
+-------+-------------------+---------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|3      |2026-05-14 23:14:48|CREATE OR REPLACE TABLE AS SELECT|{partitionBy -> [], clusterBy -> [], description -> NULL, isManaged -> true, properties -> {"delta.parquet.compression.codec":"zstd","delta.enableDeletionVectors":"tr

In [0]:
#now time travel to version 0 

df_v0 = spark.read.format("delta").option("versionAsOf",0).table("employee_grades")
df_v0.show()

+-----+-------+------+------+
| name|country|salary| grade|
+-----+-------+------+------+
|Alice|    UAE|  5000|Junior|
|  Bob|    KSA|  7000|Senior|
|Carol|    UAE|  4500|Junior|
+-----+-------+------+------+



In [0]:
#now lets time travel to version 1 

df_v1= spark.read.format("delta").option("versionAsOf",1).table("employee_grades")
df_v1.show()

+-----+-------+------+------+
| name|country|salary| grade|
+-----+-------+------+------+
|Alice|    UAE|  5000|Junior|
|  Bob|    KSA|  7000|Senior|
|Carol|    UAE|  4500|Junior|
|David|    UAE|  9000|Senior|
+-----+-------+------+------+



In [0]:
df_v2= spark.read.format("delta").option("versionAsOf",2).table("employee_grades")
df_v2.show()

+-----+-------+------+------+
| name|country|salary| grade|
+-----+-------+------+------+
|Alice|    UAE|  5000|Junior|
|  Bob|    KSA|  7000|Senior|
|Carol|    UAE|  4500|Junior|
|David|    UAE|  9000|Senior|
|David|    UAE|  9000|Senior|
+-----+-------+------+------+



In [0]:
#now lets the current state
spark.read.format("delta").table("employee_grades").show()

+-----+-------+------+------+
| name|country|salary| grade|
+-----+-------+------+------+
|  Bob|    KSA|  7000|Senior|
|Carol|    UAE|  4500|Junior|
|Alice|    UAE|  5000|Junior|
|David|    UAE|  9000|Senior|
+-----+-------+------+------+

